# Equity Duration Notebook

This notebook constructs several **duration** measures for STOXX Europe 600 constituents:

1. **Operating cash-flow timing duration** (undiscounted)
2. **Discounted cash-flow duration** with fixed discount rate \(r = 12.5\%\)
3. **Discounted cash-flow duration** with firm-specific **CAPM cost of equity**
4. **Empirical (interest-rate sensitivity) duration** using daily changes in **2Y EUR OIS** (RIC `EUREON2Y=`)

> **Note on cash flows:** `CFPSMean_*` in this project is *cash flow from operations per share* (CFO per share), not net payouts to equity holders.


In [ ]:
import lseg.data as ld
import pandas as pd
import numpy as np

EQ = "CAPP.PA"

In [4]:
path = '/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Data/STOXX600Constituents202601.xlsx'

df_const = pd.read_excel(path)

import pandas as pd

# RIC-Spalte finden
ric_col = next(c for c in df_const.columns if c.upper() == "RIC" or "RIC" in c.upper())

# Company Name finden
name_col = next(
    c for c in df_const.columns
    if "COMPANY" in c.upper()
)

# Country of Headquarters finden
country_col = next(
    c for c in df_const.columns
    if "COUNTRY" in c.upper() and ("HEAD" in c.upper())
    or c.upper() == "COUNTRY"
)

# Market Cap finden
cap_col = next(
    c for c in df_const.columns
    if "MARKET" in c.upper() and ("CAP" in c.upper())
    or c.upper() == "MARKET"
)

# Neue, reduzierte Tabelle
df_basic = df_const[[ric_col, name_col, country_col, cap_col]].copy()

# Einheitliche Spaltennamen
df_basic.columns = ["RIC", "CompanyName", "CountryHQ", "MarketCap"]

# Aufräumen
df_basic = (
    df_basic
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

print(df_basic.head())
print("Number of firms:", len(df_basic))

rics = df_basic["RIC"].tolist()

       RIC         CompanyName               CountryHQ     MarketCap
0    A2.MI             A2A SpA                 Italien   7610.187355
1    AAF.L   Airtel Africa PLC  Vereinigtes Königreich  15313.262664
2   AAK.ST       AAK AB (publ)                Schweden   6142.691574
3    AAL.L  Anglo American PLC  Vereinigtes Königreich  42523.693525
4  AALB.AS         Aalberts NV             Niederlande   3222.012813
Number of firms: 600


## Risk Free Rate $r_f$

In [35]:
import lseg.data as ld

ld.open_session()

hist = ld.get_history(
    universe="DEGR10YT=RR",
    fields=["TR.AskYield"],
    start="1999-01-01",
    end="2025-12-31",
    interval="daily"
)

ld.close_session()

assert not hist.empty, "Bund yield series is empty"

# Spalte umbenennen
hist = hist.rename(columns={"Ask Yield": "Yield"})

# In Dezimalform
hist["Yield"] = hist["Yield"] / 100

# Durchschnitt über gesamten Zeitraum
rf_avg_1999_2025 = hist["Yield"].mean()

print(f"Average 10Y German Bund yield (1999–2025): {rf_avg_1999_2025:.4%}")

Average 10Y German Bund yield (1999–2025): 1.7708%


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


## Market Rate $r_m$

## Beta $\beta$

In [ ]:
# ld.open_session()

# beta_df = ld.get_data(
#     universe=[EQ], 
#     fields=["TR.BetaFiveYear"]
# )

# ld.close_session()

# # Beta-Spalte robust finden
# beta_col = next(c for c in beta_df.columns if "Beta" in c)

# # Beta als float extrahieren
# beta_value = float(beta_df[beta_col].iloc[0])

# print(f"Beta for {EQ}: {beta_value:.3f}")

In [37]:
import numpy as np
import pandas as pd

def get_beta_5y(ric):
    beta_df = ld.get_data(
        universe=[ric],
        fields=["TR.BetaFiveYear"]
    )

    if beta_df is None or beta_df.empty:
        return np.nan

    beta_col = next((c for c in beta_df.columns if "Beta" in c), None)
    if beta_col is None:
        # falls LSEG den exakten Feldnamen als Spalte zurückgibt
        beta_col = "TR.BetaFiveYear" if "TR.BetaFiveYear" in beta_df.columns else None

    if beta_col is None:
        return np.nan

    beta = beta_df[beta_col].iloc[0]
    if beta is None:
        return np.nan

    # Beta ist i.d.R. bereits "unitless" (nicht in %)
    try:
        return float(beta)
    except Exception:
        return np.nan


def coe_capm(beta, rf=0.025, erp=0.05):
    """
    rf und erp als Dezimalzahlen (2.5% = 0.025).
    """
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


ld.open_session()

betas = []
coes = []

rf = 0.025   # Beispiel: 2.5% risk-free
erp = 0.05   # Beispiel: 5% equity risk premium

for ric in df_basic["RIC"]:
    try:
        b = get_beta_5y(ric)
    except Exception:
        b = np.nan

    betas.append(b)
    coes.append(coe_capm(b, rf=rf, erp=erp))

ld.close_session()

df_basic["BETA_5Y"] = betas
df_basic["COE_CAPM"] = coes

print(df_basic[["RIC", "CompanyName", "BETA_5Y", "COE_CAPM"]].head())
print("Missing BETA:", df_basic["BETA_5Y"].isna().sum())
print(df_basic["BETA_5Y"].describe())
print(df_basic["COE_CAPM"].describe())

       RIC         CompanyName   BETA_5Y  COE_CAPM
0    A2.MI             A2A SpA  0.880204  0.069010
1    AAF.L   Airtel Africa PLC  1.607657  0.105383
2   AAK.ST       AAK AB (publ)  0.632843  0.056642
3    AAL.L  Anglo American PLC  1.825913  0.116296
4  AALB.AS         Aalberts NV  1.327833  0.091392
Missing BETA: 15
count    585.000000
mean       1.019760
std        0.450330
min       -0.238246
25%        0.704184
50%        0.993445
75%        1.309207
max        2.441229
Name: BETA_5Y, dtype: float64
count    585.000000
mean       0.075988
std        0.022517
min        0.013088
25%        0.060209
50%        0.074672
75%        0.090460
max        0.147061
Name: COE_CAPM, dtype: float64


## Current Price $P_0$

In [12]:
def get_year_end_price(ric, year=2025):
    hist = ld.get_history(
        universe=ric,
        fields=["TR.PriceClose"],
        start=f"{year}-12-01",
        end=f"{year}-12-31",
        interval="daily"
    )
    if hist is None or hist.empty:
        return np.nan, None

    price_col = next(c for c in hist.columns if "Price Close" in c)
    price = float(hist[price_col].iloc[-1])
    date = hist.index[-1].date()
    return price, date

ld.open_session()

prices = []
dates = []

for ric in df_basic["RIC"]:
    try:
        p, d = get_year_end_price(ric, year=2025)
    except Exception:
        p, d = np.nan, None
    prices.append(p)
    dates.append(d)

ld.close_session()

df_basic["Price_2025_12_31"] = prices
df_basic["PriceDate_2025_12_31"] = dates

print(df_basic[["RIC", "CompanyName", "CountryHQ", "Price_2025_12_31", "PriceDate_2025_12_31"]].head())
print("Missing prices:", df_basic["Price_2025_12_31"].isna().sum())

       RIC         CompanyName               CountryHQ  Price_2025_12_31  \
0    A2.MI             A2A SpA                 Italien              2.31   
1    AAF.L   Airtel Africa PLC  Vereinigtes Königreich            355.20   
2   AAK.ST       AAK AB (publ)                Schweden            263.80   
3    AAL.L  Anglo American PLC  Vereinigtes Königreich           3085.00   
4  AALB.AS         Aalberts NV             Niederlande             28.06   

  PriceDate_2025_12_31  
0           2025-12-30  
1           2025-12-31  
2           2025-12-30  
3           2025-12-31  
4           2025-12-31  
Missing prices: 2


## Return on Equity

In [14]:
def get_roe(ric):
    roe_df = ld.get_data(
        universe=[ric],
        fields=["TR.F.ReturnAvgTotEqPct(period=FY0)"]
    )

    if roe_df is None or roe_df.empty:
        return np.nan

    # ROE-Spalte robust finden
    roe_col = next(
        c for c in roe_df.columns
        if "Return on Average Total Equity" in c
    )

    roe = roe_df[roe_col].iloc[0]

    if roe is None:
        return np.nan

    # LSEG liefert Prozent → in Dezimalform umwandeln
    return float(roe) / 100

import pandas as pd

ld.open_session()

roe_values = []

for ric in df_basic["RIC"]:
    try:
        roe = get_roe(ric)
    except Exception:
        roe = np.nan
    roe_values.append(roe)

ld.close_session()

df_basic["ROE_FY0"] = roe_values

print(
    df_basic[
        ["RIC", "CompanyName", "CountryHQ", "ROE_FY0"]
    ].head()
)

print("Missing ROE values:", df_basic["ROE_FY0"].isna().sum())
print(df_basic["ROE_FY0"].describe())

       RIC         CompanyName               CountryHQ   ROE_FY0
0    A2.MI             A2A SpA                 Italien  0.165109
1    AAF.L   Airtel Africa PLC  Vereinigtes Königreich  0.129261
2   AAK.ST       AAK AB (publ)                Schweden  0.189808
3    AAL.L  Anglo American PLC  Vereinigtes Königreich -0.089310
4  AALB.AS         Aalberts NV             Niederlande  0.073169
Missing ROE values: 12
count    588.000000
mean       0.150052
std        0.197391
min       -1.489826
25%        0.074911
50%        0.129454
75%        0.194284
max        2.565778
Name: ROE_FY0, dtype: float64


In [15]:
# missing = df_basic.loc[df_basic["ROE_FY0"].isna(), ["RIC", "CompanyName", "CountryHQ"]]
# print(missing)

# print(df_basic.sort_values("ROE_FY0").head(10)[["RIC","CompanyName","ROE_FY0"]])
# print(df_basic.sort_values("ROE_FY0", ascending=False).head(10)[["RIC","CompanyName","ROE_FY0", "CountryHQ"]])

## Cashflows Per Share (Estimates)

In [16]:
ld.open_session()

years = [2025, 2026, 2027, 2028, 2029]
rows = []

for y in years:
    df_y = ld.get_data(
        universe=[EQ],
        fields=["TR.FiscalYear", "TR.CFPSMean"],
        parameters={"SDate": f"{y}-12-01", "EDate": f"{y}-12-01", "FRQ": "FY"}
    )
    df_y["RequestedFY"] = y
    rows.append(df_y)

df = pd.concat(rows, ignore_index=True)

# CFPSMean-Spalte robust finden
cfps_col = next(c for c in df.columns if "Cash Flow Per Share" in c and "Mean" in c)

df = df.rename(columns={
    "Instrument": "RIC",
    cfps_col: "CFPSMean"
})

# numerisch machen
df["CFPSMean"] = pd.to_numeric(df["CFPSMean"], errors="coerce")
df["FY"] = df["RequestedFY"].astype("Int64")

# Duplikate pro RIC × FY bereinigen
df = (
    df.sort_values(["FY", "CFPSMean"], ascending=[True, False])
      .drop_duplicates(subset=["RIC", "FY"], keep="first")
      .sort_values("FY")
      .reset_index(drop=True)
)

print(df[["RIC", "FY", "CFPSMean"]])

ld.close_session()

       RIC    FY  CFPSMean
0  CAPP.PA  2025  13.79333
1  CAPP.PA  2026  15.36857
2  CAPP.PA  2027  16.36167
3  CAPP.PA  2028  16.05333
4  CAPP.PA  2029     16.69


In [17]:
import numpy as np
import pandas as pd
import lseg.data as ld

years = [2025, 2026, 2027, 2028, 2029]
rics = df_basic["RIC"].dropna().astype(str).tolist()

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

ld.open_session()

# Wir bauen je FY eine Spalte in df_basic
for y in years:
    col_name = f"CFPSMean_FY{str(y)[-2:]}"
    df_basic[col_name] = np.nan  # default

    # je nach System/Entitlement sind 100-300 RICs pro Call stabil
    for batch in chunks(rics, 200):
        df_y = ld.get_data(
            universe=batch,
            fields=["TR.CFPSMean"],
            parameters={"SDate": f"{y}-12-01", "EDate": f"{y}-12-01", "FRQ": "FY"}
        )

        if df_y is None or df_y.empty:
            continue

        # CFPSMean-Spalte finden (Label ist "Cash Flow Per Share - Mean" o.ä.)
        cfps_col = next(c for c in df_y.columns if "Cash Flow Per Share" in c and "Mean" in c)

        # Harmonisieren
        df_y = df_y.rename(columns={"Instrument": "RIC", cfps_col: col_name})
        df_y[col_name] = pd.to_numeric(df_y[col_name], errors="coerce")

        # Merge zurück in df_basic
        df_basic = df_basic.merge(df_y[["RIC", col_name]], on="RIC", how="left", suffixes=("", "_new"))
        df_basic[col_name] = df_basic[col_name].combine_first(df_basic[f"{col_name}_new"])
        df_basic.drop(columns=[f"{col_name}_new"], inplace=True)

ld.close_session()

print(df_basic[["RIC", "CompanyName", "CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]].head())
print("Missing FY25:", df_basic["CFPSMean_FY25"].isna().sum())

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_17683/2504028089.py:39:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a 

       RIC         CompanyName  CFPSMean_FY25  CFPSMean_FY26  CFPSMean_FY27  \
0    A2.MI             A2A SpA        0.57333        0.53667        0.57667   
1    AAF.L   Airtel Africa PLC          0.395          0.469         0.6565   
2   AAK.ST       AAK AB (publ)          10.58          16.16       15.77667   
3    AAL.L  Anglo American PLC        3.65286          5.178        5.57833   
4  AALB.AS         Aalberts NV           4.07           4.68           4.95   

   CFPSMean_FY28  CFPSMean_FY29  
0           0.58           0.57  
1            0.7           0.81  
2           20.7           <NA>  
3            5.5           3.68  
4           <NA>           <NA>  
Missing FY25: 95


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


## 1. Macaulay Equity Duration

$$ D = \frac{\sum_{t=1}^T t \cdot \frac{CF_t}{(1+r)^t}}{P} + (T + \frac{(1+r)}{r})\cdot \frac{(P- \sum^T_{t=1}\frac{CF_t}{(1+r)^t})}{P}$$

I use analyst consensus forecasts of operating cash flows per share.
Accordingly, the duration measure captures the timing of operating cash flow generation rather than net equity payouts.

While Dechow, Sloan and Soliman (2002) define equity duration based on net cash distributions to equity holders, I rely on analyst forecasts of operating cash flows per share due to data availability. The resulting measure captures the timing of operating cash flow generation relative to firm valuation.

In [38]:
def equity_duration_from_cfps(row, cf_cols, price_col, r_value=None, r_col=None, min_cf=2):
    """
    Implied equity duration nach Dechow/Sloan/Soliman:
    D = sum(t*PV(CF_t))/P + (T+(1+r)/r) * (P - sum(PV(CF_t)))/P

    row: eine Zeile aus df
    cf_cols: Liste CFPS-Spalten (t=1..T)
    price_col: Spalte für Preis P (oder MarketCap, siehe Hinweis)
    r_value: fixer r (z.B. 0.125) ODER
    r_col: Spaltenname mit firmenspezifischem r (z.B. COE_CAPM)
    """
    P = row.get(price_col)

    # r bestimmen
    if r_col is not None:
        r = row.get(r_col)
    else:
        r = r_value

    if P is None or not np.isfinite(P) or P <= 0:
        return np.nan
    if r is None or not np.isfinite(r) or r <= 0:
        return np.nan

    # CFPS einlesen (NaN raus)
    CF = np.array([row.get(c) for c in cf_cols], dtype="object")
    CF = pd.to_numeric(pd.Series(CF), errors="coerce").to_numpy(dtype=float)
    CF = CF[np.isfinite(CF)]

    T = len(CF)
    if T < min_cf:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    pv_cf = CF / (1.0 + r) ** t
    pv_sum = pv_cf.sum()

    term1 = (t * pv_cf).sum() / P
    term2 = (T + (1.0 + r) / r) * ((P - pv_sum) / P)

    return term1 + term2


cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]
for c in cf_cols:
    df_basic[c] = pd.to_numeric(df_basic[c], errors="coerce")

df_basic["Duration_r125"] = df_basic.apply(
    equity_duration_from_cfps,
    axis=1,
    cf_cols=cf_cols,
    price_col="Price_2025_12_31",
    r_value=0.125
)

df_basic["COE_CAPM"] = pd.to_numeric(df_basic["COE_CAPM"], errors="coerce")

df_basic["Duration_CAPM"] = df_basic.apply(
    equity_duration_from_cfps,
    axis=1,
    cf_cols=cf_cols,
    price_col="Price_2025_12_31",
    r_col="COE_CAPM"
)

print(df_basic[["RIC","CompanyName","Price_2025_12_31","COE_CAPM","Duration_r125","Duration_CAPM"]].head())
print(df_basic[["Duration_r125","Duration_CAPM"]].describe())
print("Missing Duration_r125:", df_basic["Duration_r125"].isna().sum())
print("Missing Duration_CAPM:", df_basic["Duration_CAPM"].isna().sum())

# Rang-Korrelation (oft relevanter als Pearson)
print("Spearman corr:", df_basic[["Duration_r125","Duration_CAPM"]].corr(method="spearman"))


       RIC         CompanyName  Price_2025_12_31  COE_CAPM  Duration_r125  \
0    A2.MI             A2A SpA              2.31  0.069010       4.200683   
1    AAF.L   Airtel Africa PLC            355.20  0.105383      13.936659   
2   AAK.ST       AAK AB (publ)            263.80  0.056642      11.178861   
3    AAL.L  Anglo American PLC           3085.00  0.116296      13.939346   
4  AALB.AS         Aalberts NV             28.06  0.091392       8.148887   

   Duration_CAPM  
0       2.723758  
1      15.413305  
2      18.528855  
3      14.533526  
4       9.656375  
       Duration_r125  Duration_CAPM
count     503.000000     489.000000
mean       10.074652      14.138083
std         3.169770       7.107983
min       -15.233057     -22.583553
25%         9.128122      11.701555
50%        10.644028      14.002913
75%        11.976625      16.778085
max        16.666729      81.208508
Missing Duration_r125: 97
Missing Duration_CAPM: 111
Spearman corr:                Duration_r125  D

## 2. Cash-Flow Timing (Discount-Rate–Free) Duration

$$ 
D_i \;=\;  \frac{\sum_{t=1}^T t\cdot C_t}{\sum_{t=1}^T C_t} \quad\text{(undiscounted)}
$$

In [32]:
def timing_duration_undiscounted(row, cf_cols, min_cf=2):
    CF = np.array([row.get(c) for c in cf_cols], dtype="object")
    CF = pd.to_numeric(pd.Series(CF), errors="coerce").to_numpy(dtype=float)
    CF = CF[np.isfinite(CF)]

    T = len(CF)
    if T < min_cf:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    den = CF.sum()
    if den <= 0:
        return np.nan

    return (t * CF).sum() / den

df_basic["Duration_undiscounted"] = df_basic.apply(
    timing_duration_undiscounted,
    axis=1,
    cf_cols=cf_cols,
    min_cf=2
)

In [33]:
pearson = df_basic["Duration"].corr(
    df_basic["Duration_undiscounted"],
    method="pearson"
)

spearman = df_basic["Duration"].corr(
    df_basic["Duration_undiscounted"],
    method="spearman"
)

print(f"Pearson correlation:  {pearson:.3f}")
print(f"Spearman correlation: {spearman:.3f}")

from scipy.stats import pearsonr, spearmanr

df_corr = df_basic[["Duration", "Duration_undiscounted"]].dropna()

r_p, p_p = pearsonr(df_corr["Duration"], df_corr["Duration_undiscounted"])
r_s, p_s = spearmanr(df_corr["Duration"], df_corr["Duration_undiscounted"])

print(f"Pearson r = {r_p:.3f}, p-value = {p_p:.4g}")
print(f"Spearman ρ = {r_s:.3f}, p-value = {p_s:.4g}")

Pearson correlation:  -0.041
Spearman correlation: 0.037
Pearson r = -0.041, p-value = 0.3651
Spearman ρ = 0.037, p-value = 0.4124


## 3. Empirical Duration via Interest-Rate Sensitivities

In [51]:
ld.open_session()

def get_rate_history_try_fields(ric, start="2015-01-01", end="2025-12-31",
                                fields=("TR.Yield", "TR.MIDPRICE", "TR.Last", "TR.BID", "TR.ASK", "TR.Close")):
    last_err = None
    for f in fields:
        try:
            df = ld.get_history(
                universe=[ric],
                fields=[f],
                start=start,
                end=end,
                interval="daily"
            )
            if df is not None and not df.empty:
                return f, df
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"History failed for {ric}. Last error: {last_err}")

ld.close_session()

ld.open_session()

field_used, y_df = get_rate_history_try_fields(
    "EUREON2Y=",
    start="2015-01-01",
    end="2025-12-31"
)

print("Field used:", field_used)
print(y_df.head())

ld.close_session()

Field used: TR.MIDPRICE
EUREON2Y=   Mid Price
Date                 
2014-12-31     -0.077
2015-01-02     -0.089
2015-01-05     -0.085
2015-01-06     -0.086
2015-01-07     -0.085


LSEG: tägliche Preise für RICs holen (batched)

In [53]:
import numpy as np
import pandas as pd

def chunk_list(x, n=50):
    for i in range(0, len(x), n):
        yield x[i:i+n]

def get_prices_daily_for_rics(rics, start="2015-01-01", end="2025-12-31",
                             fields=("TR.PriceClose", "TR.Close", "TR.MIDPRICE")):
    """
    Versucht pro Batch mehrere Preis-Felder; nimmt das erste Feld, das Daten liefert.
    Gibt long-format zurück: date, RIC, price
    """
    all_out = []
    last_err = None

    for batch in chunk_list(list(rics), n=40):  # 40 ist meist safe; ggf. anpassen
        got_batch = False

        for f in fields:
            try:
                df = ld.get_history(
                    universe=batch,
                    fields=[f],
                    start=start,
                    end=end,
                    interval="daily"
                )
                if df is None or df.empty:
                    continue

                # df ist häufig wide mit Date als Index und Spalten je RIC
                df_w = df.copy()
                if not isinstance(df_w.index, pd.DatetimeIndex):
                    # manchmal ist Date eine Spalte
                    date_col = next((c for c in df_w.columns if "date" in str(c).lower()), None)
                    if date_col is not None:
                        df_w[date_col] = pd.to_datetime(df_w[date_col])
                        df_w = df_w.set_index(date_col)
                    else:
                        df_w.index = pd.to_datetime(df_w.index)

                # Wide -> long
                df_long = df_w.reset_index().melt(id_vars=[df_w.index.name or "Date"], var_name="RIC", value_name="price")
                df_long = df_long.rename(columns={df_w.index.name or "Date": "date"})
                df_long["date"] = pd.to_datetime(df_long["date"])
                df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")
                df_long = df_long.dropna(subset=["price"])

                all_out.append(df_long)
                got_batch = True
                break
            except Exception as e:
                last_err = e
                continue

        if not got_batch:
            print("Batch failed (skipped):", batch[:3], "...", "err:", last_err)

    if not all_out:
        raise RuntimeError(f"No price data retrieved. Last error: {last_err}")

    return pd.concat(all_out, ignore_index=True)

In [54]:
ld.open_session()

rics = df_basic["RIC"].dropna().unique().tolist()

prices = get_prices_daily_for_rics(
    rics,
    start="2015-01-01",
    end="2025-12-31",
    fields=("TR.PriceClose", "TR.Close", "TR.MIDPRICE")
)

ld.close_session()

print(prices.head())
print(prices.columns)

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcastin

        date    RIC   price
0 2015-01-02  A2.MI  0.8355
1 2015-01-05  A2.MI   0.808
2 2015-01-06  A2.MI   0.796
3 2015-01-07  A2.MI  0.7915
4 2015-01-08  A2.MI   0.827
Index(['date', 'RIC', 'price'], dtype='object')


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


Daily returns aus Preisen berechnen

In [55]:
prices = prices.sort_values(["RIC", "date"])
prices["ret"] = prices.groupby("RIC")["price"].pct_change()
rets_daily = prices.dropna(subset=["ret"])[["date", "RIC", "ret"]].copy()

print(rets_daily.head())

        date    RIC       ret
1 2015-01-05  A2.MI -0.032914
2 2015-01-06  A2.MI -0.014851
3 2015-01-07  A2.MI -0.005653
4 2015-01-08  A2.MI  0.044852
5 2015-01-09  A2.MI -0.020556


Marktindex (mkt_ret)

In [56]:
ld.open_session()

# Beispiel-Kandidat; wenn nicht resolvable, sag mir den Error, dann finden wir den richtigen RIC bei dir
idx_ric = ".STOXX"  # ggf. anpassen!
field_used, idx_df = get_rate_history_try_fields(
    idx_ric,
    start="2015-01-01",
    end="2025-12-31",
    fields=("TR.PriceClose", "TR.Close", "TR.MIDPRICE")
)

ld.close_session()

idx = idx_df.reset_index()
idx.columns = ["date", "mkt_price"]
idx["date"] = pd.to_datetime(idx["date"])
idx["mkt_price"] = pd.to_numeric(idx["mkt_price"], errors="coerce")
idx = idx.dropna().sort_values("date")
idx["mkt_ret"] = idx["mkt_price"].pct_change()
idx = idx.dropna(subset=["mkt_ret"])[["date", "mkt_ret"]]

rets_daily = rets_daily.merge(idx, on="date", how="left")

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/.venv/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [57]:
# y_df ist dein Output von get_history (Index=Date, Spalte=Mid Price)
rate_df = y_df.copy().reset_index()

# Spalten robust umbenennen
rate_df.columns = ["date", "y"]  # date, rate in percent

rate_df["date"] = pd.to_datetime(rate_df["date"])
rate_df["y"] = pd.to_numeric(rate_df["y"], errors="coerce")

rate_df = rate_df.dropna(subset=["y"]).sort_values("date")

# Δy in Prozentpunkten (bp = 0.01, 10bp = 0.10)
rate_df["dy"] = rate_df["y"].diff()

rates_daily = rate_df[["date", "dy"]].dropna()

print(rates_daily.head())
print(rates_daily["dy"].describe())

def estimate_rate_beta_duration(rets_daily, rates_daily, min_obs=200):
    df = rets_daily.copy()
    df["date"] = pd.to_datetime(df["date"])
    rates_daily = rates_daily.copy()
    rates_daily["date"] = pd.to_datetime(rates_daily["date"])

    df = df.merge(rates_daily, on="date", how="inner")

    # excess return (optional)
    if "mkt_ret" in df.columns:
        df["rx"] = df["ret"] - df["mkt_ret"]
    else:
        df["rx"] = df["ret"]

    out = []
    for ric, g in df.groupby("RIC"):
        g = g.dropna(subset=["rx", "dy"])
        if len(g) < min_obs:
            continue

        x = g["dy"].to_numpy(float)     # percent points
        y = g["rx"].to_numpy(float)

        X = np.column_stack([np.ones_like(x), x])
        alpha, b = np.linalg.lstsq(X, y, rcond=None)[0]

        out.append({
            "RIC": ric,
            "beta_rate_2yOIS": b,
            "D_emp_2yOIS": -b,
            "n_obs": len(g)
        })

    return pd.DataFrame(out)

dur_emp = estimate_rate_beta_duration(rets_daily, rates_daily, min_obs=200)

df_basic = df_basic.merge(dur_emp, on="RIC", how="left")

print(df_basic[["RIC", "D_emp_2yOIS", "beta_rate_2yOIS", "n_obs"]].head())
print(df_basic["D_emp_2yOIS"].describe())

# Sollte i.d.R. positiv sein: längeres Timing -> stärkere Zins-Sensitivität
print(df_basic[["Duration_r125", "D_emp_2yOIS"]].corr(method="spearman"))

# Missingness
print("Share with D_emp:", df_basic["D_emp_2yOIS"].notna().mean())

        date     dy
1 2015-01-02 -0.012
2 2015-01-05  0.004
3 2015-01-06 -0.001
4 2015-01-07  0.001
5 2015-01-08   0.01
count      2846.0
mean     0.000765
std      0.035152
min        -0.348
25%        -0.008
50%           0.0
75%       0.00875
max         0.215
Name: dy, dtype: Float64
       RIC  D_emp_2yOIS  beta_rate_2yOIS   n_obs
0    A2.MI     0.048466        -0.048466  2792.0
1    AAF.L    -0.032444         0.032444  1644.0
2   AAK.ST     0.009515        -0.009515  2762.0
3    AAL.L    -0.018182         0.018182  2776.0
4  AALB.AS    -0.018158         0.018158  2813.0
count    596.000000
mean      -0.001753
std        0.037219
min       -0.148862
25%       -0.021383
50%        0.000588
75%        0.020099
max        0.116649
Name: D_emp_2yOIS, dtype: float64
               Duration_r125  D_emp_2yOIS
Duration_r125        1.00000      0.18395
D_emp_2yOIS          0.18395      1.00000
Share with D_emp: 0.9933333333333333


The empirical duration is positively but moderately correlated with the baseline operating cash-flow duration (Spearman ρ ≈ 0.18), consistent with the notion that realized interest-rate sensitivities reflect not only discounting effects but also growth and risk-related channels.

## Finale Tabelle

In [62]:
cols = [
    "RIC",
    "CompanyName",
    "CountryHQ",
    "MarketCap",
    "Price_2025_12_31",

    # Profitabilität / Risiko
    "ROE_FY0",
    "BETA_5Y",
    "COE_CAPM",

    # Analyst Cashflows
    "CFPSMean_FY25",
    "CFPSMean_FY26",
    "CFPSMean_FY27",
    "CFPSMean_FY28",
    "CFPSMean_FY29",

    # Duration Measures
    "Duration_r125",
    "Duration_CAPM",
    "Duration_undiscounted",
    "D_emp_2yOIS"
]

df_results = df_basic[cols].copy()

df_results["ROE_FY0"] = df_results["ROE_FY0"] * 100
df_results["COE_CAPM"] = df_results["COE_CAPM"] * 100

df_results = df_results.rename(columns={
    "Price_2025_12_31": "Price (Dec 31, 2025)",
    "ROE_FY0": "ROE (%)",
    "COE_CAPM": "Cost of Equity CAPM (%)",
    "BETA_5Y": "Beta (5Y)",

    "CFPSMean_FY25": "CFPS FY25",
    "CFPSMean_FY26": "CFPS FY26",
    "CFPSMean_FY27": "CFPS FY27",
    "CFPSMean_FY28": "CFPS FY28",
    "CFPSMean_FY29": "CFPS FY29",

    "Duration_r125": "Duration (r = 12.5%)",
    "Duration_CAPM": "Duration (CAPM r)",
    "Duration_undiscounted": "Duration (undiscounted)",
    "D_emp_2yOIS": "Empirical Duration (2Y OIS)"
})

df_results_rounded = df_results.round({
    "MarketCap": 2,
    "Price (Dec 31, 2025)": 2,
    "ROE (%)": 2,
    "Cost of Equity CAPM (%)": 2,
    "Beta (5Y)": 2,

    "CFPS FY25": 2,
    "CFPS FY26": 2,
    "CFPS FY27": 2,
    "CFPS FY28": 2,
    "CFPS FY29": 2,

    "Duration (r = 12.5%)": 2,
    "Duration (CAPM r)": 2,
    "Duration (undiscounted)": 2,
    "Empirical Duration (2Y OIS)": 2
})

df_results_rounded.sort_values("MarketCap", ascending=False).head(20)

,RIC,CompanyName,CountryHQ,MarketCap,"Price (Dec 31, 2025)",ROE (%),Beta (5Y),Cost of Equity CAPM (%),CFPS FY25,CFPS FY26,CFPS FY27,CFPS FY28,CFPS FY29,Duration (r = 12.5%),Duration (CAPM r),Duration (undiscounted),Empirical Duration (2Y OIS)
57,ASML.AS,ASML Holding NV,Niederlande,393698.46,921.40,43.68,2.12,13.12,22.02,27.49,36.27,47.21,52.61,12.53,12.23,3.44,0.01
343,LVMH.PA,LVMH Moet Hennessy Louis Vuitton SE,Frankreich,316187.84,645.00,19.64,1.42,9.59,33.91,35.26,37.39,47.19,<NA>,11.15,13.00,2.64,0.00
440,ROG.S,Roche Holding AG,Schweiz,296035.54,328.20,26.47,1.03,7.65,24.48,25.44,27.21,28.74,29.67,10.76,13.74,3.10,0.02
382,NOVN.S,Novartis AG,Schweiz,258020.21,109.60,26.28,0.79,6.43,9.26,9.45,10.07,11.69,14.57,10.17,13.95,3.23,0.00
457,SAPG.DE,SAP SE,Deutschland,254011.86,208.35,7.06,1.04,7.72,7.54,9.01,10.69,12.27,15.42,12.04,15.70,3.35,0.02
70,AZN.L,AstraZeneca PLC,Vereinigtes Königreich,253387.46,13790.00,17.59,0.86,6.79,9.8,10.76,12.11,12.55,16.74,13.97,20.66,3.25,0.01
264,HSBA.L,HSBC Holdings PLC,Vereinigtes Königreich,236625.46,1173.80,12.99,1.24,8.68,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,-0.07
263,HRMS.PA,Hermes International SCA,Frankreich,225891.51,2122.00,28.47,1.40,9.50,43.96,49.3,59.58,70.23,<NA>,12.19,14.46,2.70,0.02
371,NESN.S,Nestle SA,Schweiz,205994.10,78.74,30.58,0.93,7.16,5.63,5.74,6.08,6.76,<NA>,10.58,14.71,2.58,0.02
487,SIEGn.DE,Siemens Aktiengesellschaft,Deutschland,201698.65,239.15,13.37,1.37,9.33,13.96,15.32,17.29,38.42,<NA>,10.41,12.18,2.94,-0.02


In [21]:
import lseg.data as ld

ld.open_session()

df = ld.get_data(
    universe=["SAPG.DE"],
    fields=["TR.CFPSMean(period=FY0)", "TR.CFPSMean(period=FY1)", "TR.CFPSMean(period=FY2)", "TR.CFPSMean(period=FY3)", "TR.CFPSMean(period=FY4)"],
    parameters={
        "FRQ": "FY",
        "SDate": "2000-12-31",
        "EDate": "2000-12-31"
    }
)

ld.close_session()
print(df)

  Instrument  Cash Flow Per Share - Mean  Cash Flow Per Share - Mean  \
0    SAPG.DE                     0.58244                     0.63631   

   Cash Flow Per Share - Mean  Cash Flow Per Share - Mean  \
0                     0.92745                      1.2363   

   Cash Flow Per Share - Mean  
0                     1.45667  
